# Notebook 04 — Iterative Model Improvement (Part IV)

**Objective:** Apply at least ONE cycle of theory-driven improvements following the framework:

> **Baseline → Experimental Settings → Controlled Modification → Evaluation → Analysis → Conclusion**

## Improvement Strategies Investigated

| Cycle | Technique | DL Principle | Motivation from NB02/03 |
|-------|-----------|--------------|-------------------------|
| 1 | L2 Regularisation (weight decay) | Prevent overfitting | Val–train gap observed in baseline |
| 2 | Mosaic + MixUp augmentation | Improve generalisation | Small-object under-representation |
| 3 | Combined best config | Ensemble of improvements | Best-of-above |


In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from scipy.ndimage import uniform_filter1d

sns.set_theme(style='whitegrid')

DATA_CFG    = ROOT / 'configs' / 'visdrone.yaml'
PROJECT_DIR = ROOT / 'results'
DEVICE      = '0'   # change to 'cpu' if needed

# Load baseline for reference
baseline_json = ROOT / 'results' / 'baseline' / 'baseline_metrics.json'
with open(baseline_json) as f:
    BASELINE = json.load(f)

print('Baseline reference:')
for k, v in BASELINE.items():
    print(f'  {k}: {v}')

## Helper

In [ ]:
def run_improvement(checkpoint, exp_name, train_kwargs, description=''):
    """Run a single improvement cycle and return metrics."""
    print(f"\n{'═'*58}")
    print(f"  Improvement: {exp_name}")
    if description:
        print(f"  Motivation: {description}")
    print(f"{'═'*58}")

    model = YOLO(checkpoint)
    t0 = time.time()
    results = model.train(
        data     = str(DATA_CFG),
        project  = str(PROJECT_DIR / 'experiments'),
        name     = exp_name,
        device   = DEVICE,
        exist_ok = True,
        plots    = True,
        verbose  = False,
        **train_kwargs,
    )
    elapsed = time.time() - t0

    rd = results.results_dict
    m = {
        'experiment': exp_name,
        'mAP50':      round(float(rd.get('metrics/mAP50(B)', 0)), 4),
        'mAP50_95':   round(float(rd.get('metrics/mAP50-95(B)', 0)), 4),
        'precision':  round(float(rd.get('metrics/precision(B)', 0)), 4),
        'recall':     round(float(rd.get('metrics/recall(B)', 0)), 4),
        'train_min':  round(elapsed / 60, 1),
        'save_dir':   str(results.save_dir),
    }
    with open(Path(results.save_dir) / 'metrics.json', 'w') as f:
        json.dump(m, f, indent=2)

    delta_map50 = m['mAP50'] - BASELINE['mAP50']
    print(f"  mAP50={m['mAP50']:.4f} (Δ={delta_map50:+.4f} vs baseline)")
    return m


def load_csv(save_dir):
    csv = Path(save_dir) / 'results.csv'
    if not csv.exists(): return None
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    return df

## Cycle 1 — L2 Regularisation (Weight Decay)

**Principle**: L2 regularisation adds a penalty term `λ||w||²` to the loss, discouraging large weights and reducing overfitting.

**Baseline setting**: `weight_decay=0.0005`  
**Modified settings**: `weight_decay=0.005` (10×) and `weight_decay=0.001`

**Expected**: Higher weight decay → less overfitting → smaller val–train gap, possibly slightly lower final mAP.

In [ ]:
c1_wd_001 = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c1_wd_0001',
    description='Increased weight decay from 0.0005 to 0.001 to reduce overfitting',
    train_kwargs=dict(epochs=50, imgsz=640, batch=16, lr0=0.01, lrf=0.01,
                      weight_decay=0.001, patience=20, workers=4),
)

c1_wd_005 = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c1_wd_0005',
    description='Heavy weight decay = 0.005 to test strong regularisation',
    train_kwargs=dict(epochs=50, imgsz=640, batch=16, lr0=0.01, lrf=0.01,
                      weight_decay=0.005, patience=20, workers=4),
)

## Cycle 2 — Data Augmentation Strategy

**Principle**: Stronger augmentation artificially increases dataset diversity, reducing overfitting and improving generalisation to unseen drone viewpoints.

**Ultralytics augmentation flags**: `mosaic`, `mixup`, `flipud`, `degrees`, `scale`

**Modification**: Enable enhanced augmentation via `augment=True` and increased mosaic probability.

In [ ]:
c2_augment = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c2_augment',
    description='Enable Ultralytics augment=True (mosaic + mixup + flips) to reduce overfitting',
    train_kwargs=dict(epochs=50, imgsz=640, batch=16, lr0=0.01, lrf=0.01,
                      weight_decay=0.0005, patience=20, workers=4,
                      augment=True, mosaic=1.0, mixup=0.1, flipud=0.1),
)

## Cycle 3 — Combined Best Config

Combine:
- Best weight decay from Cycle 1
- Augmentation from Cycle 2
- Cosine LR schedule (smoother convergence)
- Increased image size if E1 from NB03 showed improvement

In [ ]:
c3_combined = run_improvement(
    checkpoint='yolo11s.pt',
    exp_name='c3_combined_best',
    description='Combined: wd=0.001, augment=True, cos_lr=True, imgsz=1280',
    train_kwargs=dict(epochs=50, imgsz=1280, batch=8, lr0=0.01, lrf=0.01,
                      weight_decay=0.001, patience=20, workers=4,
                      augment=True, mosaic=1.0, mixup=0.1,
                      cos_lr=True),
)

## Quantitative Comparison Table

In [ ]:
all_cycles = [
    {'experiment': 'Baseline',         **{k: BASELINE[k] for k in ['mAP50','mAP50_95','precision','recall']}},
    {**c1_wd_001, 'change': 'wd=0.001'},
    {**c1_wd_005, 'change': 'wd=0.005'},
    {**c2_augment, 'change': 'augment=True'},
    {**c3_combined, 'change': 'combined best'},
]

comp_df = pd.DataFrame(all_cycles)
comp_df['ΔmAP50'] = (comp_df['mAP50'] - BASELINE['mAP50']).round(4)

print('\nImprovement Comparison Table')
print('=' * 70)
cols = ['experiment', 'mAP50', 'mAP50_95', 'precision', 'recall', 'ΔmAP50']
print(comp_df[cols].to_string(index=False))

comp_df[cols].to_csv(PROJECT_DIR / 'experiments' / 'improvement_comparison.csv', index=False)
print('\nSaved to results/experiments/improvement_comparison.csv')

## Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.tab10.colors

names = comp_df['experiment']
x = range(len(names))

# mAP50 comparison
bars = axes[0].bar(x, comp_df['mAP50'], color=colors[:len(names)])
axes[0].axhline(BASELINE['mAP50'], color='gray', linestyle='--', label=f'Baseline ({BASELINE["mAP50"]:.4f})')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('mAP@50')
axes[0].set_title('Improvement Cycles — mAP@50')
axes[0].legend(fontsize=8)
for bar, v in zip(bars, comp_df['mAP50']):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=8)

# Delta mAP50
deltas = comp_df['ΔmAP50']
bar_colors = ['green' if d >= 0 else 'red' for d in deltas]
bars = axes[1].bar(x, deltas, color=bar_colors)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('ΔmAP50 vs Baseline')
axes[1].set_title('Improvement vs Baseline')
for bar, v in zip(bars, deltas):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 v + (0.001 if v >= 0 else -0.003),
                 f'{v:+.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Iterative Improvement Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'experiments' / 'improvement_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Overfitting Analysis: Baseline vs Best Improved

In [ ]:
def get_loss_gap(save_dir, train_col='train/box_loss', val_col='val/box_loss'):
    df = load_csv(save_dir)
    if df is None: return None, None
    tc = next((c for c in [train_col, 'train/box_om'] if c in df.columns), None)
    vc = next((c for c in [val_col, 'val/box_om'] if c in df.columns), None)
    if tc and vc:
        return df[tc].values, df[vc].values
    return None, None

baseline_dir = ROOT / 'results' / 'baseline'
best_dir = c3_combined.get('save_dir')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (run_dir, label) in zip(axes, [(baseline_dir, 'Baseline'), (best_dir, 'Combined (C3)')]):
    tr, vl = get_loss_gap(run_dir)
    if tr is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    ep = range(len(tr))
    ax.fill_between(ep, tr, vl, alpha=0.15, color='tomato', label='Gap')
    ax.plot(ep, uniform_filter1d(tr, size=5), color='royalblue', label='Train', linewidth=2)
    ax.plot(ep, uniform_filter1d(vl, size=5), color='tomato', linestyle='--', label='Val', linewidth=2)
    gap = np.mean(vl[-10:] - tr[-10:])
    ax.set_title(f'{label} — Final Gap: {gap:.4f}', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Box Loss')
    ax.legend(fontsize=8)

plt.suptitle('Train–Val Gap: Baseline vs Best Improvement', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'experiments' / 'gap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Analysis & Conclusions

### Cycle 1 — Weight Decay
- **Justification**: Baseline showed a val–train gap, indicating some overfitting. L2 regularisation adds a penalty `λ||w||²` to discourage large weights.
- **Observed**: (fill in after running)
- **Conclusion**: (was overfitting reduced? did mAP improve/decline?)

### Cycle 2 — Augmentation
- **Justification**: VisDrone has relatively few training images for a challenging task. Augmentation (mosaic, mixup) creates synthetic training variety.
- **Observed**: (fill in)
- **Conclusion**: (did augmentation improve generalisation?)

### Cycle 3 — Combined
- **Justification**: Combine the best-performing modifications.
- **Observed**: (fill in)
- **Conclusion**: (are improvements additive or do they interact?)

The best-performing configuration from this notebook is used in **Notebook 05** to provide the primary YOLO11 result for the multi-version comparison.